In [1]:
from auto import AUTOCommands as ac  
from auto import runAUTO as ra  
import matplotlib.pyplot as plt  
from pathlib import Path  

output = Path("output") 


-------------------------------------------------------------
Could not import plotting modules, plotting will be disabled.
This is probably because Tkinter is not enabled in your Python installation.
-------------------------------------------------------------



# Homogeneous 1-D bifurcation plot

In [2]:
runner = ra.runAUTO()  # Create a fresh AUTO runner to manage the calculation.

# Run continuation for 1D plot. 
reference = ac.run("common_model", runner=runner, PAR={"beta": 0},  
                   NMX=8000, NPR=1000, DSMAX=.005, EPSS=1e-8,  
                   RL0=0, RL1=.15, UZSTOP={"mu": [0, .15]})  

# Find labelled point
bp = reference("BP1")           # Get the first branch point: the recovery threshold in this model.
lp = reference("LP1")           # Get the first fold (limit point): the collapse threshold in this model.
print(f"Recovery: {bp['mu']}")  # Print harvest at recovery 
print(f"Collapse: {lp['mu']}")  # Print harvest at collapse 
print(f"Hysteresis width: {lp['mu'] - bp['mu']}")

Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    27  BP    2   1.14286E-01   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    35  UZ    3   1.50000E-01   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2   373  LP    4   1.32335E-01   7.71689E+00   3.27910E-01   5.43997E+00   2.72857E-01   3.27910E-01   5.43997E+00   2.72857E-01
   2  1000        5   2.71289E-02   5.89831E+00   1.11244E+00   3.76267E+00   1.41414E+00   1.11244E+00   3.76267E+00   1.41414E+00
   2  1128  UZ    6   1.84465E-15   5.87943E+0

gfortran -g -fopenmp -O common_model.o -o common_model.exe /opt/auto-07p/lib/*.o


Starting common_model ...



  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      


   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00


...

   2  1000        7   4.37662E-02   1.43328E+01  -6.32634E-01   1.01094E+01  -3.37757E-01  -6.32634E-01   1.01094E+01  -3.37757E-01


   2  2000        8   1.22630E-03   1.93142E+01  -1.19831E+00   1.35950E+01  -5.09157E-01  -1.19831E+00   1.35950E+01  -5.09157E-01


   2  2038  UZ    9   2.59094E-16   1.95035E+01  -1.22060E+00   1.37273E+01  -5.14615E-01  -1.22060E+00   1.37273E+01  -5.14615E-01


 Total Time    0.287E+00


common_model ... done
Recovery: 0.11428571429
Collapse: 0.13233512487
Hysteresis width: 0.01804941057999998
Adult predator biomass at collapse: 0.32791024293


In [3]:
fig, ax = plt.subplots(figsize=(8, 4.5)) # Create the figure and plotting area                       

for branch in reference:                 # Work through each equilibrium branch returned by AUTO.
    adult = branch["PP"].copy()          # Copy pelagic adult predator biomass
    adult[adult < -1e-8] = float("nan")  # remove negative values
    color = "black"                      # Start with black for a predator-free branch.
    if max(branch["PP"]) > 1e-8:         # Check whether the branch contains positive adult predator biomass.
        color = "#C44E52"                # Use this red colour for a branch containing predators.
        
    first = 0                            # Start drawing at the first point (Python counts from zero).

    # branch.stability() returns information about the branch stability. It gives the ending point number of each section using its sign to encode stability
    # negative --> stable
    # positive --> unstable
    # Example: branch.stability() returns [27, -35] - the stability of one particular branch
    # The unstable section ends at AUTO point 27.
    # The stable section ends at AUTO point 35.
    # The plotting code shares point 27 between the two sections so they connect.
    # The result is:
    # [0:27]  draws AUTO points 1–27 as a dashed line (exclude the number after the colon)
    # [26:35] draws AUTO points 27–35 as a solid line (exclude the number after the colon)
    # Note that 
    
    for endpoint in branch.stability():   # Read the signed endpoints of sections with constant stability.
        last = abs(endpoint)              # Remove the sign to get the end position for an array slice. (does not change endpoint) 
        if last > 1:                      # Skip an initial section that contains only one point.
            style = "--"                  # Choose a dashed line for an unstable section.
            if endpoint < 0:              # A negative endpoint marks a stable section in AUTO.
                style = "-"               # Choose a solid line for that stable section.
            ax.plot(                      # Draw the selected section of the branch, using harvest on the horizontal axis and adult predator biomass on the vertical axis.
                branch["mu"][first:last], # Horizontal coordinates: harvest.
                adult[first:last],        # Vertical coordinates: biomass.
                color=color,              # Use the colour chosen earlier.
                linestyle=style           # Use the solid or dashed style chosen earlier.
            )
            first = last - 1              # Share the boundary point with the next section to keep the curve connected.

ax.plot(bp["mu"], bp["PP"][0], "ko", label="Recovery")                           # Mark recovery with a black circle; ko means black plus a circle marker.
ax.plot(lp["mu"], lp["PP"][0], "o", color="#C44E52", label="Collapse")           # Mark collapse with a red circle
ax.plot([], [], color="gray", label="Stable")                                    # Add a grey solid-line example to the legend without drawing extra data.
ax.plot([], [], "--", color="gray", label="Unstable")                            # Add a grey dashed-line example showing the unstable line style.
ax.set(xlim=(0, .15), xlabel=r"Adult predator harvest ($\mu$)",                  # Set the harvest range and x-axis label; the r string preserves the math notation.
       ylabel=r"Adult predator biomass ($A$)", title="Homogeneous reference")  # Set the biomass label and the figure title.
ax.legend()                                                                      # Show the labels supplied above in a legend.
fig.tight_layout()                                                               # Adjust spacing so the labels fit around the plot.
fig.savefig(output / "homogeneous_bifurcation.png", dpi=600)                     # Save a 600-dpi PNG; save location --> output folder

# Experiment 1

In [4]:
runner = ra.runAUTO() 
fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), sharey=True)  # Create two side-by-side panels with a shared vertical scale.

# runs the indented code twice: once for beta = 0, then once for beta = 5.
# enumerate(...) pairs each value with its position in the list, counting from zero
# validate: print("panel =", panel, "beta =", beta)

for panel, beta in enumerate([0, 5]):                                               
    recovery = ac.run(bp, runner=runner, ICP=["mu", "deltar"], ISW=2,               
                      PAR={"beta": beta}, DS=-.001, DSMIN=1e-6, DSMAX=.005,  
                      NMX=24000, NPR=1000, RL0=-.25, RL1=.9, UZSTOP={"deltar": 1})  # stop at productivity variation = 1
    collapse = ac.run(lp, runner=runner, ICP=["mu", "deltar"], ISW=2,  
                      PAR={"beta": beta}, DS=.001, DSMIN=1e-6, DSMAX=.005,  
                      NMX=24000, NPR=1000, RL0=-.25, RL1=.9, UZSTOP={"deltar": 1})  # stop at productivity variation = 1

    r0, r1 = recovery["mu"][0], recovery["mu"][-1]                          # store recovery harvest at first point [0] and last point [-1]
    c0, c1 = collapse["mu"][0], collapse["mu"][-1]                          # store collapse harvest at first point [0] and last point [-1]
    print(f"\nExperiment 1, beta = {beta}: contrast 0 -> 1")                # Print a heading; \n starts a new line and the f string inserts beta.

    print(f"Recovery: {r0} -> {r1} ({100 * (r1 / r0 - 1)}%)")               # Print the initial/final recovery thresholds and their percentage change.
    print(f"Collapse: {c0} -> {c1} ({100 * (c1 / c0 - 1)}%)")               # Print the initial/final collapse thresholds and their percentage change.
    print(f"Width: {c0-r0} -> {c1-r1} ({100 * ((c1-r1) / (c0-r0) - 1)}%)")  # Print initial/final hysteresis widths and their percentage change.

    axes[panel].plot(collapse["deltar"], collapse["mu"], color="#C44E52", label="Collapse")  # Plot collapse against productivity contrast as a red solid curve.
    axes[panel].plot(recovery["deltar"], recovery["mu"], "k--", label="Recovery")            # Plot recovery as a black dashed curve; k-- specifies black and dashed.
    axes[panel].set(xlim=(0, 1), xlabel=r"Productivity contrast ($\Delta r$)")               # Set this panel’s contrast range and x-axis label.
    
axes[0].set_title("Random movement")                          # Label the left panel, where beta = 0.
axes[1].set_title("Fitness-dependent movement")               # Label the right panel, where beta = 5.
axes[0].set_ylabel(r"Adult predator harvest ($\mu$)")         # Label the shared harvest scale on the left panel.
axes[0].legend()                                              # Show a legend
fig.suptitle("Experiment 1: Variation in Productivity")           
fig.tight_layout()                                            # Adjust the panel spacing to fit the labels and title.
fig.savefig(output / "experiment_1_thresholds.png", dpi=600)  # Save the two-panel figure as a 600-dpi PNG.

Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltar    
   2   231  UZ   10   1.08813E-01   9.78249E+00   3.72389E-28   6.64708E+00   4.78081E-28   3.64591E-28   7.17729E+00   1.00000E+00

 Total Time    0.317E-01
common_model ... done
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltar    
   4   220  UZ   10   1.27102E-01   7.96423E+00   3.39814E-01   5.39479E+00   2.77761E-01   3.39776E-01   5.82588E+00   1.00000E+00

 Total Time    0.298E-01
common_model ... done

Experiment 1, beta = 0: contrast 0 -> 1
Recovery: 0.11428571429 -> 0.10881343686 (-4.788242751070449%)
Collapse: 0.13233512487 -> 0.12710224301 (-3.9542652528121502%)
Width: 0.01804941057999998 -> 0.018288806150000003 (1.3263345577904317%)
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM        

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltar    


   2   231  UZ   10   1.08813E-01   9.78249E+00   3.72389E-28   6.64708E+00   4.78081E-28   3.64591E-28   7.17729E+00   1.00000E+00


 Total Time    0.318E-01


common_model ... done
Starting common_model ...



...

   4  1000       10   1.81524E-01   1.07936E+01   7.61338E-02   3.99753E+00   1.33738E-01   7.25372E-02   1.00246E+01   6.92372E-01


   4  1450  UZ   11   2.15483E-01   1.25524E+01   5.53470E-02   3.26337E+00   1.09982E-01   5.23192E-02   1.21201E+01   1.00000E+00


 Total Time    0.198E+00


common_model ... done



Experiment 1, beta = 5: contrast 0 -> 1
Recovery: 0.11428571429 -> 0.21281662057 (+86.215%)
Collapse: 0.13233512487 -> 0.21548310958 (+62.831%)
Width: 0.01804941057999998 -> 0.002666489009999995 (-85.227%)


# Experiment #2

In [5]:
# Same template for experiment #2 - see above for comments. 

runner = ra.runAUTO()
fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), sharey=True)

for panel, beta in enumerate([0, 5]):
    recovery = ac.run(bp, runner=runner, ICP=["mu", "deltas"], ISW=2,
                      PAR={"beta": beta}, DS=.001, DSMIN=1e-6, DSMAX=.005,
                      NMX=24000, NPR=1000, RL0=-.25, RL1=.9, UZSTOP={"deltas": 1})
    collapse = ac.run(lp, runner=runner, ICP=["mu", "deltas"], ISW=2,
                      PAR={"beta": beta}, DS=.001, DSMIN=1e-6, DSMAX=.005,
                      NMX=24000, NPR=1000, RL0=-.25, RL1=.9, UZSTOP={"deltas": 1})

    r0, r1 = recovery["mu"][0], recovery["mu"][-1]
    c0, c1 = collapse["mu"][0], collapse["mu"][-1]
    print(f"\nExperiment 2, beta = {beta}: contrast 0 -> 1")
    print(f"Recovery: {r0} -> {r1} ({100 * (r1 / r0 - 1)}%)")
    print(f"Collapse: {c0} -> {c1} ({100 * (c1 / c0 - 1)}%)")
    print(f"Width: {c0-r0} -> {c1-r1} ({100 * ((c1-r1) / (c0-r0) - 1)}%)")

    axes[panel].plot(collapse["deltas"], collapse["mu"], color="#C44E52", label="Collapse")
    axes[panel].plot(recovery["deltas"], recovery["mu"], "k--", label="Recovery")
    axes[panel].set(xlim=(0, 1), xlabel=r"Interaction Strength ($\Delta a$)")

axes[0].set_title("Random movement")
axes[1].set_title("Fitness-dependent movement")
axes[0].set_ylabel(r"Adult predator harvest ($\mu$)")
axes[0].legend()
fig.suptitle("Experiment 2: Variation in Interaction Strength")
fig.tight_layout()
fig.savefig(output / "experiment_2_thresholds.png", dpi=600)

Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltas    
   2   204  UZ   10   1.28649E-01   9.42809E+00  -1.21842E-58   6.66667E+00  -1.00720E-58  -1.21368E-58   6.66667E+00   1.00000E+00

 Total Time    0.285E-01
common_model ... done
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltas    
   4   209  UZ   10   1.43984E-01   7.90254E+00   2.89943E-01   5.61023E+00   2.69200E-01   2.89030E-01   5.53911E+00   1.00000E+00

 Total Time    0.291E-01
common_model ... done

Experiment 2, beta = 0: contrast 0 -> 1
Recovery: 0.11428571429 -> 0.1286490493 (12.56791813327871%)
Collapse: 0.13233512487 -> 0.14398355481 (8.802220840039944%)
Width: 0.01804941057999998 -> 0.015334505509999985 (-15.041516496989116%)
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          P

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltas    


   2   204  UZ   10   1.28649E-01   9.42809E+00  -1.21842E-58   6.66667E+00  -1.00720E-58  -1.21368E-58   6.66667E+00   1.00000E+00


 Total Time    0.282E-01


common_model ... done
Starting common_model ...



...

common_model ... done
Starting common_model ...



  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP          deltas    


   4   523  UZ   10   5.93482E-01   9.38679E+00   7.37659E-03   6.65703E+00   3.15968E-02   6.50578E-03   6.61776E+00   1.00000E+00


 Total Time    0.769E-01


common_model ... done

Experiment 2, beta = 5: contrast 0 -> 1
Recovery: 0.11428571429 -> 0.59330869798 (+419.145%)
Collapse: 0.13233512487 -> 0.59348221165 (+348.469%)
Width: 0.01804941057999998 -> 0.00017351367000006945 (-99.039%)


# Experiment 3

In [6]:
runner = ra.runAUTO() 

# preperation for the experiment 3
# This finds a suitable starting equilibrium when both habitat contrast are 0.4 
# deltas = 0.4, and we vary deltar from 0 to 0.4 
# harvest fixed at 0
initial = ac.run("common_model", runner=runner, ICP=["deltar"], PAR={"beta": 0, "deltas": .4},  
                 DS=.01, DSMIN=1e-6, DSMAX=.02, EPSS=1e-8,  
                 RL0=-.1, RL1=.5, NMX=2000, ISW=1, ILP=0, ISP=0, MXBF=0,  
                 UZSTOP={"deltar": .4})                                                 # AUTO saves a solution when deltar reaches 0.4 and STOP. This solution has the label UZ1

# Finds the harvest levels for recovery and collapse for fixed habitat heterogeneity; harvest fixed at 0
# Restart from that saved equilibrium and now vary harvest (mu).
heterogeneous = ac.run(initial("UZ1"), runner=runner, ICP=["mu"],                       
                       DS=.0005, DSMAX=.05, RL0=-.25, RL1=.9, NMX=8000, NPR=1000,  
                       ISW=1, ILP=1, ISP=2, MXBF=10, UZSTOP={"mu": [-.25, .9]})  

# UZR tells us to collect data at different beta values checkpoints {0,1,2,3,4,5} --> saves us UZ
recovery_beta = ac.run(heterogeneous("BP1"), runner=runner, ICP=["mu", "beta"], ISW=2,  
                       DS=.001, DSMIN=1e-6, DSMAX=.02, NMX=24000, NPR=1000, 
                       UZR={"beta": [1, 2, 3, 4]}, UZSTOP={"beta": 5})                
collapse_beta = ac.run(heterogeneous("LP1"), runner=runner, ICP=["mu", "beta"], ISW=2,  
                       DS=.001, DSMIN=1e-6, DSMAX=.02, NMX=24000, NPR=1000,  
                       UZR={"beta": [1, 2, 3, 4]}, UZSTOP={"beta": 5})  


Starting common_model ...

  BR    PT  TY  LAB     deltar       L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    25  UZ    2   4.00000E-01   9.48496E+00  -4.77207E-25   6.60296E+00  -3.71537E-25  -4.64459E-25   6.80921E+00  -3.39995E-25

 Total Time    0.390E-02
common_model ... done
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1    10  BP    3   1.14782E-01   9.48496E+00  -3.83472E-21   6.60296E+00  -2.84401E-21  -3.82880E-21   6.80921E+00  -2.64261E-21
   1    26  UZ    4   9.00000E-01   9.48496E+00   8.47656E-45   6.60296E+00  -1.75162E-46   8.24409E-45   6.80921E+00   4.92644E-47

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP     

  BR    PT  TY  LAB     deltar       L2-NORM          PL            FL            JL            PP            FP            JP      


   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00


   1    25  UZ    2   4.00000E-01   9.48496E+00  -4.77207E-25   6.60296E+00  -3.71537E-25  -4.64459E-25   6.80921E+00  -3.39995E-25


 Total Time    0.378E-02


...

   5   218  UZ   13   2.11963E-01   9.62558E+00   5.60807E-02   5.26738E+00   1.07344E-01   5.32124E-02   8.05537E+00   3.00000E+00


   5   270  UZ   14   2.25278E-01   9.68735E+00   5.24366E-02   5.10035E+00   1.06827E-01   4.95106E-02   8.23496E+00   4.00000E+00


   5   321  UZ   15   2.32046E-01   9.71987E+00   5.11241E-02   4.96794E+00   1.06937E-01   4.81756E-02   8.35339E+00   5.00000E+00


 Total Time    0.495E-01


common_model ... done


In [7]:
recovery_points = [heterogeneous("BP1")] + list(recovery_beta("UZ"))  # Combine the beta-zero recovery point with saved points at beta = 1 through 5.
collapse_points = [heterogeneous("LP1")] + list(collapse_beta("UZ"))  # Build the corresponding list of collapse points at the same beta values.
print("Beta   Recovery       Collapse       Hysteresis width")        # Print headings for the four columns in the results table.
for i in range(6):                                                    # Loop through the six list positions, numbered 0 to 5.
    r = recovery_points[i]["mu"]                                      # Read harvest at recovery for this beta value.
    c = collapse_points[i]["mu"]                                      # Read harvest at collapse for the same beta value.
    print(f"{i}      {r}    {c}    {c-r}")                            # Print beta, both thresholds, and their difference.
r0, r1 = recovery_beta["mu"][0], recovery_beta["mu"][-1]              # Read recovery harvest at the start (beta = 0) and end (beta = 5).
c0, c1 = collapse_beta["mu"][0], collapse_beta["mu"][-1]              # Read collapse harvest at the start and end.
print(f"Recovery change: {100 * (r1 / r0 - 1):+.3f}%")                # Print the percentage change in recovery; +.3f shows the sign and three decimals.
print(f"Collapse change: {100 * (c1 / c0 - 1):+.3f}%")                # Print the percentage change in collapse.
print(f"Hysteresis-width change: {100 * ((c1-r1) / (c0-r0) - 1)}%")   # Print the percentage change in hysteresis width.


Beta   Recovery       Collapse       Hysteresis width
0      0.11478233967    0.13292016125    0.018137821580000005
1      0.14110280447    0.15120137475    0.010098570279999997
2      0.18170656328    0.18554354329    0.0038369800099999907
3      0.20920365844    0.21196331844    0.002759659999999997
4      0.22265588814    0.22527763304    0.0026217449000000004
5      0.22948957723    0.23204634604    0.0025567688099999986
Recovery change: +99.935%
Collapse change: +74.576%
Hysteresis-width change: -85.90366103932092%


In [8]:
fig, ax = plt.subplots(figsize=(8, 4.5))  
ax.plot(collapse_beta["beta"], collapse_beta["mu"], color="#C44E52", label="Collapse")     # Draw collapse harvest against beta as a red solid line.
ax.plot(recovery_beta["beta"], recovery_beta["mu"], "k--", label="Recovery")               # Draw recovery harvest against beta as a black dashed line.
ax.set(xlim=(0, 5), xlabel=r"Senstivity to fitness-dependent movement ($\beta$)",                              # Show beta from 0 to 5 and label the horizontal axis.
       ylabel=r"Adult predator harvest ($\mu$)")  # Label the vertical harvest axis and describe the plot.
ax.legend()                                                                                # Show the recovery and collapse legend.
fig.suptitle("Experiment 3: Fixed Habitat Variation and Fitness-dependent Movement")                                         # Add the experiment title above the plot.
fig.tight_layout()                                                                         # Adjust spacing so the titles and axis labels fit.
fig.savefig(output / "experiment_3_thresholds.png", dpi=600)                               # Save the figure as a 600-dpi PNG in the output folder.


`reference` The complete continuation result from your ac.run() - including branches and labelled points

`branch` One equilibrium curve (sequence of equilibrium points forming a curve)

`bp` The equilibrium at the first detected branch point (one point)

`lp` The equilibrium at the first detected limit point (one point)